# HealthConnect Clinic — Week 7
# Analytics Testing, Refinement & End-to-End Validation
## Week 7 focus

Week 7 continues directly from the Week 6 work.

In Week 6, the Python analysis was deepened and important combined-variable relationships were investigated. The Power BI dashboard was also improved, but the combined findings were not fully incorporated into the dashboard.

Therefore, Week 7 focuses on:

1. Testing the accuracy and consistency of the Week 6 analytical outputs.
2. Validating the five dashboard KPIs.
3. Testing the important combined-variable findings against the underlying dataset.
4. Refining the existing Week 6 dashboard by incorporating validated combined findings.
5. Retesting the refined dashboard and documenting the results.
6. Validating the relevant cross-track dependency with Data Science.
7. Assessing readiness for Week 8 final integration and presentation.

# 1. Import Libraries

In [1]:
# Import libraries

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 2. Load Dataset

In [2]:
# Load the existing cleaned dataset

df = pd.read_csv("HealthConnect_Appointment_Data_Cleaned.csv")

# 3. Matching Week 6 Data Handling

In [3]:
# Match the Week 6 date handling

df["booking_date"] = pd.to_datetime(df["booking_date"])
df["appointment_date"] = pd.to_datetime(df["appointment_date"])

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset shape: (5000, 19)
Columns:
['appointment_id', 'patient_id', 'gender', 'age', 'age_group', 'appointment_type', 'booking_date', 'appointment_date', 'appointment_day', 'appointment_time', 'booking_lead_days', 'previous_appointments', 'previous_no_shows', 'reminder_sent', 'reminder_channel', 'distance_to_clinic_km', 'waiting_time_minutes', 'appointment_outcome', 'lead_time_group']


# 4. Data Readiness Check

In [4]:
# Data readiness checks
# This is a validation check, not a new cleaning exercise.

print("Duplicate appointment IDs:", df["appointment_id"].duplicated().sum())
print("\nOutcome values:")
print(df["appointment_outcome"].value_counts(dropna=False))

Duplicate appointment IDs: 0

Outcome values:
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64


## Result Explanation
The appointment outcomes remain consistent with the expected dataset structure:
This confirms that the underlying dataset is intact and that the Week 7 validation can proceed using the same cleaned data used for the Week 6 analysis.

# 5. KPI Validation

The Week 6 dashboard used five KPIs:

1. Total Appointments
2. No-Show Rate
3. Attended Rate
4. Average Booking Lead Time 
5. Reminder Coverage

The purpose of this test is to independently recalculate them from the underlying dataset and compare them with the Week 6 reported values.

In [5]:
# Recalculate the five dashboard KPIs

total_appointments = len(df)

no_show_rate = (
    (df["appointment_outcome"] == "No-Show").sum()
    / total_appointments * 100
)

attendance_rate = (
    (df["appointment_outcome"] == "Attended").sum()
    / total_appointments * 100
)

average_booking_lead_time = (
    df["booking_lead_days"].mean()
)

reminder_coverage = (
    (df["reminder_sent"] == "Yes").sum()
    / total_appointments * 100
)

kpi_validation = pd.DataFrame({
    "KPI": [
        "Total Appointments",
        "No-Show Rate",
        "Attendance Rate",
        "Average Booking Lead Time",
        "Reminder Coverage"
    ],
    "Validated Value": [
        total_appointments,
        no_show_rate,
        attendance_rate,
        average_booking_lead_time,
        reminder_coverage
    ]
})

kpi_validation

,KPI,Validated Value
0,Total Appointments,"5,000.00"
1,No-Show Rate,48.46
2,Attendance Rate,46.28
3,Average Booking Lead Time,29.64
4,Reminder Coverage,72.68


In [6]:
# Compare the recalculated KPIs with the Week 6 reported baseline

week6_kpis = {
    "Total Appointments": 5000,
    "No-Show Rate": 48.46,
    "Attendance Rate": 46.28,
    "Average Booking Lead Time": 29.64,
    "Reminder Coverage": 72.68
}

kpi_test = kpi_validation.copy()
kpi_test["Week 6 Reported Value"] = kpi_test["KPI"].map(week6_kpis)

kpi_test["Difference"] = (
    kpi_test["Validated Value"] - kpi_test["Week 6 Reported Value"]
).round(2)

kpi_test["Difference"] = kpi_test["Difference"].mask(
    kpi_test["Difference"].abs() < 0.005,
    0.0
)

kpi_test["Pass/Fail"] = np.where(
    kpi_test["Difference"].abs() <= 0.01,
    "PASS",
    "REVIEW"
)

kpi_test


,KPI,Validated Value,Week 6 Reported Value,Difference,Pass/Fail
0,Total Appointments,"5,000.00","5,000.00",0.00,PASS
1,No-Show Rate,48.46,48.46,0.00,PASS
2,Attendance Rate,46.28,46.28,0.00,PASS
3,Average Booking Lead Time,29.64,29.64,0.00,PASS
4,Reminder Coverage,72.68,72.68,0.00,PASS


## Result Explanation

The five Week 6 dashboard KPIs were independently recalculated from the dataset and compared with the previously reported values.

All five KPIs produced a **PASS** result, with differences of **0.00**.

This confirms that the Week 6 KPI calculations remain accurate and reproducible from the underlying dataset. 

 # 6. Analytical Validation — Week 6 Combined Findings

The following tests deliberately focus on the important **combined analysis from Week 6**.

This avoids repeating the full Week 5/6 EDA and instead asks:

Do the important Week 6 findings remain valid when recalculated from the underlying dataset?


## 6.1 Booking Lead Time × Previous No-Show History

Week 6 found a strong combined pattern: longer booking lead times were associated with higher no-show rates, and previous no-show history generally increased risk within lead-time groups.

The Week 6 reported rates were:

- 0–7 days: 21.77%, 33.16%, 43.59%
- 8–14 days: 31.65%, 31.85%, 47.83%
- 15–30 days: 38.40%, 46.78%, 57.62%
- 31–45 days: 49.33%, 59.61%, 62.73%
- 46–60 days: 61.68%, 73.67%, 82.11%

Columns represent previous no-show groups **0, 1 and 2+**.


In [7]:
# Order Lead_time

lead_time_order = [
    "0-7 days",
    "8-14 days",
    "15-30 days",
    "31-45 days",
    "46-60 days"
]

df["previous_no_show_group"] = df["previous_no_shows"].apply(
    lambda x: "0" if x == 0 else "1" if x == 1 else "2+"
)
df["previous_no_show_group"].value_counts()

lead_previous_test = (
    df.groupby(["lead_time_group", "previous_no_show_group"])
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

lead_previous_test["no_show_rate"] = (
    lead_previous_test["no_shows"]
    / lead_previous_test["appointments"] * 100
)

lead_previous_test["lead_time_group"] = pd.Categorical(
    lead_previous_test["lead_time_group"],
    categories=lead_time_order,
    ordered=True
)

lead_previous_test = (
    lead_previous_test
    .sort_values(["lead_time_group", "previous_no_show_group"])
    .reset_index(drop=True)
)

display(lead_previous_test.round(2))


,lead_time_group,previous_no_show_group,appointments,no_shows,no_show_rate
0,0-7 days,0,372,81,21.77
1,0-7 days,1,190,63,33.16
2,0-7 days,2+,78,34,43.59
3,8-14 days,0,376,119,31.65
4,8-14 days,1,157,50,31.85
5,8-14 days,2+,69,33,47.83
6,15-30 days,0,763,293,38.40
7,15-30 days,1,419,196,46.78
8,15-30 days,2+,151,87,57.62
9,31-45 days,0,742,366,49.33


In [8]:
# Compare the recalculated rates with the Week 6 reported rates

week6_lead_previous = pd.DataFrame(
    {
        "0": [21.77, 31.65, 38.40, 49.33, 61.68],
        "1": [33.16, 31.85, 46.78, 59.61, 73.67],
        "2+": [43.59, 47.83, 57.62, 62.73, 82.11]
    },
    index=lead_time_order
)

validated_lead_previous = (
    lead_previous_test
    .pivot(
        index="lead_time_group",
        columns="previous_no_show_group",
        values="no_show_rate"
    )
    .reindex(lead_time_order)
)

difference_lead_previous = (
    validated_lead_previous - week6_lead_previous
).round(2)

difference_lead_previous = difference_lead_previous.mask(
    difference_lead_previous.abs() < 0.005,
    0.0
)

print("Week 6 reported values:")
display(week6_lead_previous)

print("Week 7 recalculated values:")
display(validated_lead_previous.round(2))

print("Difference:")
display(difference_lead_previous)


Week 6 reported values:


,0,1,2+
0-7 days,21.77,33.16,43.59
8-14 days,31.65,31.85,47.83
15-30 days,38.40,46.78,57.62
31-45 days,49.33,59.61,62.73
46-60 days,61.68,73.67,82.11


Week 7 recalculated values:


previous_no_show_group,0,1,2+
lead_time_group,,,
0-7 days,21.77,33.16,43.59
8-14 days,31.65,31.85,47.83
15-30 days,38.40,46.78,57.62
31-45 days,49.33,59.61,62.73
46-60 days,61.68,73.67,82.11


Difference:


previous_no_show_group,0,1,2+
lead_time_group,,,
0-7 days,0.00,0.00,0.00
8-14 days,0.00,0.00,0.00
15-30 days,0.00,0.00,0.00
31-45 days,0.00,0.00,0.00
46-60 days,0.00,0.00,0.00


In [9]:
# Test the key Week 6 conclusion

highest_lead_previous = lead_previous_test.loc[
    lead_previous_test["no_show_rate"].idxmax()
]

print("Highest-risk combination:")
print("Lead time:", highest_lead_previous["lead_time_group"])
print("Previous no-show group:", highest_lead_previous["previous_no_show_group"])
print("Appointments:", highest_lead_previous["appointments"])
print("No-show rate:", round(highest_lead_previous["no_show_rate"], 2), "%")

print("\nExpected Week 6 highest-risk combination:")
print("46–60 days + 2+ previous no-shows = 82.11%")


Highest-risk combination:
Lead time: 46-60 days
Previous no-show group: 2+
Appointments: 123
No-show rate: 82.11 %

Expected Week 6 highest-risk combination:
46–60 days + 2+ previous no-shows = 82.11%


## Result Explanation

The validation confirms the Week 6 finding that **booking lead time and previous no-show history are associated with increasing no-show rates when examined together**.

No-show rates increase substantially as booking lead time becomes longer across all previous no-show groups. The highest observed rate is among appointments booked **46–60 days in advance for patients with 2+ previous no-shows**, where the no-show rate is **82.11% across 123 appointments**.

The result matches the Week 6 finding exactly, confirming that this combined pattern remains valid in Week 7.

This supports retaining **long booking lead time and previous no-show history as key risk indicators** in the dashboard and wider HealthConnect solution.


## 6.2 Booking Lead Time × Reminder Status

Week 6 found that no-show rates increased across lead-time groups for both reminder statuses.

The most important Week 6 segment was:

 **46–60 days + no reminder = 73.51% no-show rate across 302 appointments.**


In [10]:
# Compare the recalculated rates with the Week 6 reported rates

lead_reminder_test = (
    df.groupby(["lead_time_group", "reminder_sent"])
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

lead_reminder_test["no_show_rate"] = (
    lead_reminder_test["no_shows"]
    / lead_reminder_test["appointments"] * 100
)

lead_reminder_test["lead_time_group"] = pd.Categorical(
    lead_reminder_test["lead_time_group"],
    categories=lead_time_order,
    ordered=True
)

lead_reminder_test = (
    lead_reminder_test
    .sort_values(["lead_time_group", "reminder_sent"])
    .reset_index(drop=True)
)

display(lead_reminder_test.round(2))

lead_reminder_matrix = lead_reminder_test.pivot(
    index="lead_time_group",
    columns="reminder_sent",
    values="no_show_rate"
)

display(lead_reminder_matrix.round(2))


,lead_time_group,reminder_sent,appointments,no_shows,no_show_rate
0,0-7 days,No,167,50,29.94
1,0-7 days,Yes,473,128,27.06
2,8-14 days,No,161,56,34.78
3,8-14 days,Yes,441,146,33.11
4,15-30 days,No,378,176,46.56
5,15-30 days,Yes,955,400,41.88
6,31-45 days,No,358,198,55.31
7,31-45 days,Yes,900,479,53.22
8,46-60 days,No,302,222,73.51
9,46-60 days,Yes,865,568,65.66


reminder_sent,No,Yes
lead_time_group,,
0-7 days,29.94,27.06
8-14 days,34.78,33.11
15-30 days,46.56,41.88
31-45 days,55.31,53.22
46-60 days,73.51,65.66


In [11]:
# Highest lead time reminder

highest_lead_reminder = lead_reminder_test.loc[
    lead_reminder_test["no_show_rate"].idxmax()
]

print("Highest-risk lead-time/reminder segment:")
print("Lead time:", highest_lead_reminder["lead_time_group"])
print("Reminder sent:", highest_lead_reminder["reminder_sent"])
print("Appointments:", highest_lead_reminder["appointments"])
print("No-show rate:", round(highest_lead_reminder["no_show_rate"], 2), "%")

print("\nWeek 6 expected result:")
print("46–60 days + No reminder = 73.51% across 302 appointments")


Highest-risk lead-time/reminder segment:
Lead time: 46-60 days
Reminder sent: No
Appointments: 302
No-show rate: 73.51 %

Week 6 expected result:
46–60 days + No reminder = 73.51% across 302 appointments


## Result Explanation

The validation confirms that no-show rates increase progressively as booking lead time increases, regardless of whether a reminder was sent.

For appointments booked **46–60 days in advance**, the no-show rate was **73.51% when no reminder was sent**, compared with **65.66% when a reminder was sent**. The same pattern is observed across all lead-time groups.

The highest-risk segment is therefore **46–60 days + No Reminder**, with a **73.51% no-show rate across 302 appointments**.

The finding matches the Week 6 result and indicates that reminders may be a useful operational lever, particularly for appointments scheduled far in advance. However, this analysis shows an association and does not establish that reminders alone caused the difference in no-show rates.

## 6.3 Previous No-Show History × Reminder Status

Week 6 found that previous no-show history remained important when reminder status was considered.

The key Week 6 result was:

**2+ previous no-shows + no reminder = 68.24% no-show rate across 148 appointments.**


In [12]:
previous_reminder_test = (
    df.groupby(["previous_no_show_group", "reminder_sent"])
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

previous_reminder_test["no_show_rate"] = (
    previous_reminder_test["no_shows"]
    / previous_reminder_test["appointments"] * 100
)

display(previous_reminder_test.round(2))

previous_reminder_matrix = previous_reminder_test.pivot(
    index="previous_no_show_group",
    columns="reminder_sent",
    values="no_show_rate"
)

display(previous_reminder_matrix.round(2))


,previous_no_show_group,reminder_sent,appointments,no_shows,no_show_rate
0,0,No,790,359,45.44
1,0,Yes,2131,912,42.80
2,1,No,428,242,56.54
3,1,Yes,1120,586,52.32
4,2+,No,148,101,68.24
5,2+,Yes,383,223,58.22


reminder_sent,No,Yes
previous_no_show_group,,
0,45.44,42.80
1,56.54,52.32
2+,68.24,58.22


In [13]:
# Highest previous no show and reminder status

highest_previous_reminder = previous_reminder_test.loc[
    previous_reminder_test["no_show_rate"].idxmax()
]

print("Highest-risk previous-history/reminder segment:")
print("Previous no-show group:", highest_previous_reminder["previous_no_show_group"])
print("Reminder sent:", highest_previous_reminder["reminder_sent"])
print("Appointments:", highest_previous_reminder["appointments"])
print("No-show rate:", round(highest_previous_reminder["no_show_rate"], 2), "%")

print("\nWeek 6 expected result:")
print("2+ previous no-shows + No reminder = 68.24% across 148 appointments")


Highest-risk previous-history/reminder segment:
Previous no-show group: 2+
Reminder sent: No
Appointments: 148
No-show rate: 68.24 %

Week 6 expected result:
2+ previous no-shows + No reminder = 68.24% across 148 appointments


## Result Explanation

The validation shows that previous no-show history remains associated with higher no-show rates regardless of reminder status.

Among patients with **2+ previous no-shows**, the no-show rate was **68.24% when no reminder was sent** compared with **58.22% when a reminder was sent**. Patients with no previous no-shows had substantially lower rates under both reminder conditions.

The highest-risk combination is therefore **2+ previous no-shows + No Reminder**, with a **68.24% no-show rate across 148 appointments**.

This reproduces the Week 6 finding and supports using previous attendance behaviour as an important indicator when identifying appointments that may require additional support.

# 7. Analytical Validation 2 

## 7.1 Waiting Time × Appointment Outcome

In [14]:
# Test waiting time against appointment outcome

waiting_time_test = (
    df.groupby("appointment_outcome")["waiting_time_minutes"]
    .agg(["count", "mean", "median"])
    .round(2)
)

waiting_time_test

,count,mean,median
appointment_outcome,,,
Attended,2293,24.29,24.00
Cancelled,261,23.21,22.00
No-Show,2386,24.20,24.00


In [15]:
# Waiting time: targeted validation of the Week 6 waiting-time finding

waiting_bins = [0, 15, 30, 45, 60, np.inf]
waiting_labels = [
    "0-15 minutes",
    "16-30 minutes",
    "31-45 minutes",
    "46-60 minutes",
    "60+ minutes"
]

df["waiting_time_group"] = pd.cut(
    df["waiting_time_minutes"],
    bins=waiting_bins,
    labels=waiting_labels,
    include_lowest=True
)

waiting_time_test = (
    df.groupby("waiting_time_group", observed=False)
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

waiting_time_test["no_show_rate"] = (
    waiting_time_test["no_shows"]
    / waiting_time_test["appointments"] * 100
).round(2)

display(waiting_time_test)

print("Week 6 reported key values:")
print("0–15 minutes = 47.06%")
print("16–30 minutes = 49.19%")
print("31–45 minutes = 47.78%")
print("46–60 minutes = 46.21%")
print("60+ minutes = 66.67% across 3 appointments")

,waiting_time_group,appointments,no_shows,no_show_rate
0,0-15 minutes,1073,505,47.06
1,16-30 minutes,2472,1216,49.19
2,31-45 minutes,1260,602,47.78
3,46-60 minutes,132,61,46.21
4,60+ minutes,3,2,66.67


Week 6 reported key values:
0–15 minutes = 47.06%
16–30 minutes = 49.19%
31–45 minutes = 47.78%
46–60 minutes = 46.21%
60+ minutes = 66.67% across 3 appointments


## Result Explanation

The validation confirms that waiting time does not show a meaningful difference between attended and no-show appointments. Average waiting time was **24.29 minutes for attended appointments** and **24.20 minutes for no-show appointments**, a difference of only **0.09 minutes**.

The Week 7 retest also showed no clear or consistent increase in no-show rates across the waiting-time groups. Although the **60+ minutes** group recorded **66.67%**, it was based on only **3 appointments**. Overall, the Week 7 results remain consistent with the Week 6 finding that waiting time is **not a strong explanatory factor for no-show behaviour in this dataset**.


## 7.2 Distance to Clinic × Appoimntment Outcome

In [16]:
# Distance: targeted validation of the Week 6 distance finding

distance_bins = [0, 5, 10, 15, 20, np.inf]
distance_labels = ["0-5 km", "6-10 km", "11-15 km", "16-20 km", "20+ km"]

df["distance_group"] = pd.cut(
    df["distance_to_clinic_km"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=True
)

distance_test = (
    df.groupby("distance_group", observed=False)
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

distance_test["no_show_rate"] = (
    distance_test["no_shows"]
    / distance_test["appointments"] * 100
).round(2)

display(distance_test)

print("Week 6 reported key values:")
print("0–5 km = 46.45%")
print("16–20 km = 51.40%")
print("20+ km = 57.76% across 393 appointments")


,distance_group,appointments,no_shows,no_show_rate
0,0-5 km,1156,537,46.45
1,6-10 km,1692,787,46.51
2,11-15 km,1132,549,48.50
3,16-20 km,537,276,51.40
4,20+ km,393,227,57.76


Week 6 reported key values:
0–5 km = 46.45%
16–20 km = 51.40%
20+ km = 57.76% across 393 appointments


## Result Explanation

The validation confirms the Week 6 observation that no-show rates generally increase as distance from the clinic increases.

The no-show rate rises from **46.45% for patients living 0–5 km away** to **57.76% for patients living more than 20 km away**, based on **393 appointments** in the 20+ km group.

The pattern remains consistent with the Week 6 analysis. However, distance shows a less pronounced relationship with no-show behaviour than the stronger patterns observed for booking lead time and previous no-show history.

## 7.3 Age Group x Appointment Outcome

In [17]:
# Age: targeted validation of the Week 6 age finding

age_bins = [0, 24, 34, 44, 54, 64, np.inf]
age_labels = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

age_test = (
    df.groupby("age_group", observed=False)
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
      .reset_index()
)

age_test["no_show_rate"] = (
    age_test["no_shows"]
    / age_test["appointments"] * 100
).round(2)

display(age_test)

print("Week 6 reported key values:")
print("55–64 years = 50.75% across 800 appointments")
print("65+ years = 45.12% across 1,241 appointments")

,age_group,appointments,no_shows,no_show_rate
0,18-24,564,283,50.18
1,25-34,783,397,50.70
2,35-44,819,396,48.35
3,45-54,793,381,48.05
4,55-64,800,406,50.75
5,65+,1241,560,45.12


Week 6 reported key values:
55–64 years = 50.75% across 800 appointments
65+ years = 45.12% across 1,241 appointments


## Result Explanation

The age-group validation broadly reproduces the Week 6 findings. No-show rates are relatively similar across most age groups, with the **55–64 group recording 50.75% across 800 appointments** and the **65+ group recording 45.12% across 1,241 appointments**.

The differences between age groups are relatively modest compared with the stronger patterns observed for booking lead time and previous no-show history.

Therefore, age remains a **supporting descriptive factor rather than a primary risk indicator** for the Week 7 analysis.

## 8. Cross-Track Collaboration with Data Science

Following the completion of the Week 7 Data Analytics validation and Power BI refinement, the Data Analytics findings were reviewed alongside the Week 7 Data Science results.

The purpose of this collaboration was to identify the Data Science findings that were directly relevant to the Analytics output, assess how they relate to the completed Analytics analysis, and document the resulting integration decisions.

The collaboration focused on model-relevant factors, the interpretation of no-show risk, and the potential operational use of the Data Science model outputs.

Because the Data Analytics and Data Science tracks use different definitions for calculating no-show rates, their reported percentages were not treated as directly interchangeable. The comparison therefore focused on the underlying patterns, relevant segments, and integration implications.

### 8.1  Data Science Findings Relevant to Analytics

The Week 7 Data Science work included model performance validation, model refinement, probability calibration, threshold testing, segment-level validation, and feature-importance analysis.

For cross-track collaboration, the findings most relevant to the Data Analytics output were:

- booking lead time remained an important predictive factor;
- the 41–60 day lead-time band remained a model-relevant segment;
- previous no-show history remained an important predictor;
- the refined model showed improved discrimination compared with the baseline model; and
- the optimal classification threshold varied under different assumed cost ratios.

The collaboration therefore focused on the findings that could create a direct dependency for the Analytics dashboard or its interpretation.

In [18]:
# Data Science findings relevant to the Data Analytics track

ds_findings = pd.DataFrame({
    "DS Finding": [
        "Booking lead time remained a key predictive factor",
        "41–60 day lead-time band remained model-relevant",
        "Previous no-show history remained an important predictor",
        "Refined model showed improved discrimination",
        "Optimal classification threshold varied with assumed cost ratio"
    ],
    "Analytics Relevance": [
        "Corresponds to the validated lead-time analysis",
        "Corresponds to the model-relevant lead-time segment",
        "Corresponds to the validated previous no-show analysis",
        "Relevant to interpretation of the refined model",
        "Relevant to potential operational use of model risk predictions"
    ]
})

ds_findings

,DS Finding,Analytics Relevance
0,Booking lead time remained a key predictive fa...,Corresponds to the validated lead-time analysis
1,41–60 day lead-time band remained model-relevant,Corresponds to the model-relevant lead-time se...
2,Previous no-show history remained an important...,Corresponds to the validated previous no-show ...
3,Refined model showed improved discrimination,Relevant to interpretation of the refined model
4,Optimal classification threshold varied with a...,Relevant to potential operational use of model...


### 8.2 Identification of the Analytics Dependency

The Data Science findings were mapped to the Analytics work that had already been completed during Week 7.

The main dependency was the use of model-relevant appointment factors to understand and communicate the underlying appointment patterns.

The Data Analytics track had already examined:

- booking lead time;
- previous no-show history;
- booking lead time × previous no-show history;
- booking lead time × reminder status; and
- previous no-show history × reminder status.

These findings had already been incorporated into the existing Power BI dashboard.

Therefore, the cross-track activity did not require the Analytics analysis to be repeated. Instead, the collaboration established how the Data Science model findings relate to the already validated Analytics evidence.

In [19]:
# Map Data Science findings to the completed Analytics outputs

integration_map = pd.DataFrame({
    "Data Science Finding": [
        "Booking lead time",
        "41–60 day lead-time segment",
        "Previous no-show history",
        "Combined lead time × previous no-show history",
        "Threshold sensitivity"
    ],
    "Existing Analytics Output": [
        "Lead-time no-show analysis",
        "Lead-time segment analysis",
        "Previous no-show analysis",
        "Combined-variable dashboard analysis",
        "No fixed operational threshold in dashboard"
    ],
    "Integration Status": [
        "Already validated",
        "Already represented",
        "Already validated",
        "Already incorporated into dashboard",
        "Decision required"
    ]
})

integration_map

,Data Science Finding,Existing Analytics Output,Integration Status
0,Booking lead time,Lead-time no-show analysis,Already validated
1,41–60 day lead-time segment,Lead-time segment analysis,Already represented
2,Previous no-show history,Previous no-show analysis,Already validated
3,Combined lead time × previous no-show history,Combined-variable dashboard analysis,Already incorporated into dashboard
4,Threshold sensitivity,No fixed operational threshold in dashboard,Decision required


### 8.3 Metric Alignment

The Data Analytics and Data Science tracks use different definitions for calculating no-show rates.

As a result, the numerical no-show percentages reported by the two tracks are not directly interchangeable.

The Analytics rates represent observed appointment-level outcomes using the Analytics calculation approach, while the Data Science results are based on the outcome definition used for model development and evaluation.

Therefore, the cross-track comparison was based on the underlying patterns and model-relevant segments rather than requiring identical percentages.

This distinction was documented to prevent an apparent numerical difference between the two tracks from being incorrectly interpreted as a contradiction.

In [20]:
# Document the metric-alignment consideration

metric_alignment = pd.DataFrame({
    "Track": [
        "Data Analytics",
        "Data Science"
    ],
    "Purpose": [
        "Describe observed appointment-level patterns",
        "Develop and evaluate predictive model performance"
    ],
    "No_Show_Rate_Use": [
        "Observed no-show rate using Analytics definition",
        "Model-related no-show metric using Data Science definition"
    ],
    "Cross_Track_Use": [
        "Used to validate appointment patterns",
        "Used to interpret model findings"
    ]
})

metric_alignment

,Track,Purpose,No_Show_Rate_Use,Cross_Track_Use
0,Data Analytics,Describe observed appointment-level patterns,Observed no-show rate using Analytics definition,Used to validate appointment patterns
1,Data Science,Develop and evaluate predictive model performance,Model-related no-show metric using Data Scienc...,Used to interpret model findings


### 8.4 Cross-Track Integration

The Data Science model-refinement findings were considered alongside the completed Data Analytics results.

The comparison showed that the Data Science model-relevant factors had corresponding Analytics evidence. Booking lead time and previous no-show history had already been validated in the Analytics analysis, while their combined relationship had already been incorporated into the Power BI dashboard.

The integration therefore consisted of linking the model-relevant Data Science findings to the existing Analytics evidence rather than reproducing the Data Science model within the Analytics notebook.

The combined-variable dashboard views provide an Analytics-level representation of the appointment conditions associated with the model-relevant factors.

### 8.5 Data Science Threshold Analysis: Analytics Implication

Data Science evaluated different classification thresholds under different assumed cost ratios.

The results showed that the threshold selected for identifying higher-risk appointments depends on the relative cost assigned to false negatives and false positives.

Because the cost ratios used in the Data Science analysis were illustrative and actual HealthConnect operational cost information was not available, the Analytics track did not adopt any of the tested thresholds as a fixed operational rule.

The finding was therefore treated as a business dependency for future implementation rather than being converted into an unsupported dashboard rule.

In [21]:
# Data Science threshold scenarios reviewed during cross-track integration

threshold_review = pd.DataFrame({
    "Cost_Ratio": ["1:1", "2:1", "5:1"],
    "DS_Optimal_Threshold": [0.50, 0.35, 0.25],
    "Analytics_Decision": [
        "Not adopted",
        "Not adopted",
        "Not adopted"
    ]
})

threshold_review

,Cost_Ratio,DS_Optimal_Threshold,Analytics_Decision
0,1:1,0.50,Not adopted
1,2:1,0.35,Not adopted
2,5:1,0.25,Not adopted


### Cross-Track Collaboration Outcome

The Week 7 cross-track collaboration established a documented connection between the Data Science model-refinement findings and the completed Data Analytics output.

The model-relevant factors identified by Data Science had corresponding evidence in the Analytics analysis and were already represented in the refined Power BI dashboard.

The difference between the Data Analytics and Data Science no-show-rate definitions was documented as a metric-alignment consideration, and the percentages were therefore not treated as directly comparable.

The Data Science threshold analysis was also reviewed for potential Analytics implementation. No fixed operational threshold was introduced because the business cost information required to select an appropriate threshold was unavailable.

Overall, the collaboration resulted in:

**Data Science finding → Analytics dependency identified → existing Analytics evidence linked → integration decision documented → outstanding business dependency carried forward.**

In [22]:
# Final Week 7 cross-track integration status

integration_status = pd.DataFrame({
    "Integration Component": [
        "Model-relevant factors",
        "Lead-time segment",
        "Previous no-show history",
        "Combined-variable findings",
        "No-show-rate metric alignment",
        "Operational threshold"
    ],
    "Status": [
        "Integrated",
        "Integrated",
        "Integrated",
        "Already represented in dashboard",
        "Documented",
        "Deferred pending business cost information"
    ]
})

integration_status

,Integration Component,Status
0,Model-relevant factors,Integrated
1,Lead-time segment,Integrated
2,Previous no-show history,Integrated
3,Combined-variable findings,Already represented in dashboard
4,No-show-rate metric alignment,Documented
5,Operational threshold,Deferred pending business cost information
